# Phase 5 (LSI) — c-1 decoder-only: BitFit (fill the TODOs)

Multi-label statute identification: each case gets a **set** of statute IDs. Run the cells top to bottom.

- Cells marked **[ANCILLARY]** are plumbing: loading, tokenising, scoring, the writer, windowing, the loss, thresholds and the training loop (same as the c-6 Phase 5 template, with the c-1 decoder path). Nothing to change there, except `ROOT` in Cell A if your input path differs.
- **Cell E is yours**: every hyperparameter and design choice (`None`), which bias groups BitFit trains (`BITFIT_GROUPS`), and the `LSIHead`. It stops with a clear error at the first thing you haven't filled.
- Note: BitFit is **not** on the Phase 5 technique list (section 11.2.1), so this run does not count toward the required techniques; it can still count as a c-1 submission/experiment.

## Cell 0a — [ANCILLARY] imports + HF login

In [5]:
# --- Phase 1 c-1 notebook, cell 1 (verbatim) ---
from collections import Counter
from tqdm.auto import tqdm
import pickle
import re
from collections import Counter, defaultdict


# HF login (Phase 1 cell 2), needed if the dataset download asks for auth
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## Cell 0b — [ANCILLARY] tokenizer (Phase 1)

In [6]:
# --- Phase 1 c-1 notebook, cell 8 (verbatim) ---


SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]  # fixed ids 0-3, never merged

class BPETokenizer:
    def __init__(self):
        self.id_to_bytes = {}
        self.bytes_to_id = {}
        self.merges = {}
        self.merge_order = []
        self.merge_ranks = {}  # Precomputed pair -> rank for fast encoding
        self.cache = {}        # Chunk lookup cache
        self._vocab_size = 0
        
        # Splits on whitespace+word or just whitespace. 
        # Guarantees round-trip: "".join(chunks) == text
        self.split_pattern = re.compile(r'\s*\S+|\s+')

    def train(self, text: str, vocab_size: int):
        # 1. Setup base vocabulary
        for i, tok in enumerate(SPECIAL_TOKENS):
            b = tok.encode("utf-8")
            self.id_to_bytes[i] = b
            self.bytes_to_id[b] = i

        next_id = len(SPECIAL_TOKENS)
        byte_to_base_id = {}
        for value in range(256):
            b = bytes([value])
            self.id_to_bytes[next_id] = b
            self.bytes_to_id[b] = next_id
            byte_to_base_id[value] = next_id
            next_id += 1

        # 2. Split text into chunks and get frequencies
        chunks = self.split_pattern.findall(text)
        chunk_counts = Counter(chunks)
        
        # 3. Store each distinct chunk as a tuple of base byte ids
        chunk_to_ids = {
            chunk: tuple(byte_to_base_id[b] for b in chunk.encode("utf-8"))
            for chunk in chunk_counts
        }

        # 4. Initialize weighted pair counts and the pair-to-chunk index
        pair_counts = Counter()
        pair_to_chunks = defaultdict(set)
        
        def get_pairs(ids):
            return [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]

        for chunk, ids in chunk_to_ids.items():
            count = chunk_counts[chunk]
            for pair in get_pairs(ids):
                pair_counts[pair] += count
                pair_to_chunks[pair].add(chunk)

        target = vocab_size
        num_merges = target - next_id
        pbar = tqdm(total=num_merges, desc="Training BPE")

        # 5. Iteratively merge the best pair
        while next_id < target:
            if not pair_counts:
                break

            max_freq = max(pair_counts.values())
            if max_freq <= 1:
                break

            # Min tuple tie-breaker matching your original behavior
            best_pair = min(p for p, c in pair_counts.items() if c == max_freq)

            new_id = next_id
            self.merges[best_pair] = new_id
            self.merge_order.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merge_order) - 1
            
            self.id_to_bytes[new_id] = self.id_to_bytes[best_pair[0]] + self.id_to_bytes[best_pair[1]]
            self.bytes_to_id[self.id_to_bytes[new_id]] = new_id

            # 6. Update only the chunks containing the merged pair
            chunks_to_update = list(pair_to_chunks[best_pair])
            for chunk in chunks_to_update:
                old_ids = chunk_to_ids[chunk]
                count = chunk_counts[chunk]
                
                # Subtract old pair combinations
                for pair in get_pairs(old_ids):
                    pair_counts[pair] -= count
                    if pair_counts[pair] <= 0:
                        del pair_counts[pair]
                    if chunk in pair_to_chunks[pair]:
                        pair_to_chunks[pair].remove(chunk)
                
                # Apply the merge inside this chunk
                new_ids = []
                i = 0
                while i < len(old_ids):
                    if i < len(old_ids) - 1 and (old_ids[i], old_ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(old_ids[i])
                        i += 1
                new_ids = tuple(new_ids)
                chunk_to_ids[chunk] = new_ids
                
                # Add new pair combinations created by the merge
                for pair in get_pairs(new_ids):
                    pair_counts[pair] += count
                    pair_to_chunks[pair].add(chunk)

            next_id += 1
            pbar.update(1)

        pbar.close()
        self._vocab_size = next_id

    def encode(self, text: str):
        chunks = self.split_pattern.findall(text)
        out_ids = []
        
        for chunk in chunks:
            # Check the dictionary cache first
            if chunk in self.cache:
                out_ids.extend(self.cache[chunk])
                continue
            
            # Cache miss: break into bytes and process
            chunk_bytes = chunk.encode("utf-8")
            ids = [self.bytes_to_id[bytes([b])] for b in chunk_bytes]
            
            # Merge until no adjacent pair has a valid merge rank
            while len(ids) > 1:
                pairs = [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
                best_pair = None
                best_rank = float('inf')
                
                # Find the pair with the lowest rank
                for p in pairs:
                    rank = self.merge_ranks.get(p, float('inf'))
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = p
                
                if best_pair is None:
                    break  # No more valid merges for this chunk
                    
                # Apply the best merge to all occurrences in the chunk
                new_id = self.merges[best_pair]
                new_ids = []
                i = 0
                while i < len(ids):
                    if i < len(ids) - 1 and (ids[i], ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(ids[i])
                        i += 1
                ids = new_ids
            
            # Store result in cache and append
            self.cache[chunk] = ids
            out_ids.extend(ids)
            
        return out_ids

    def decode(self, ids):
        out = b"".join(self.id_to_bytes[i] for i in ids)
        return out.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self._vocab_size

    @property
    def vocab_size(self):
        return self._vocab_size


def train_tokenizer(train_text: str, vocab_size: int):
    """Train the A1 BPE tokenizer on `train_text` and return it, ready for A3."""
    tok = BPETokenizer()
    tok.train(train_text, vocab_size)
    return tok

## Cell 0c — [ANCILLARY] model classes (Phase 1 c-1)

In [7]:
# --- Phase 1 c-1 notebook, cell 21 (verbatim) ---
# ============================================================
# CELL 1 — Decoder-only Transformer (your A3 Q1 code, unchanged except the 2 marked LayerNorm lines)
# Replaces: §9 "Define The Model" (GPT2Config / GPT2LMHeadModel) and the DataCollator cell.
# ============================================================
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------- A3 Q1.1: manual causal multi-head self-attention ----------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head)
        q = q.transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head)
        k = k.transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head)
        v = v.transpose(1, 2)

        scores = q @ k.transpose(-2, -1) * self.attention_scale
        mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


# ---------- A3 Q1.2: decoder block and language model ----------
@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 1024
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 4          # == n_heads for plain MHA; unused in this version
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = False     # True = pre-norm, False = post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    pe = torch.zeros(max_len, d_model)
    for a in range(max_len):
        for b in range(d_model):
            i = b // 2
            angle = a / (10000 ** (2 * i / d_model))
            if b % 2 == 0:
                pe[a][b] = math.sin(angle)
            else:
                pe[a][b] = math.cos(angle)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))      # FIX: your A3 code used self.ln1 here (ln2 was never used)
        else:
            x = self.ln1(x + self.attn(x))     # FIX: your A3 code used self.ln2 here (ln1 was never used)
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[:T, :]
        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits

## Cell A — [ANCILLARY] load c-1 + tokenizer

In [8]:
# =====================================================================================
# CELL A — load the Phase 1 c-1 decoder and its tokenizer
# Expect <All keys matched successfully>, 110.98M params, vocab 10000, d_model 1024, 8 layers, max_seq_len 2048
# =====================================================================================
import torch, pickle, json, re, os
ROOT = "/kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only"   # TODO: the folder the find command showed (up to, not including, cs781_phase1_full)
device = torch.device("cuda")

# [c-1] checkpoint folder is legal-gpt2-final (c-6 was model_encdec); class is DecoderLM + LMConfig
tok = pickle.load(open(f"{ROOT}/cs781_phase1_full/legal-gpt2-final/bpe_tokenizer.pkl", "rb"))
ck = torch.load(f"{ROOT}/cs781_phase1_full/legal-gpt2-final/model_final.pt", map_location="cpu", weights_only=False)
cfg = LMConfig(**ck["config"])
lm = DecoderLM(cfg); print(lm.load_state_dict(ck["model"])); lm.to(device).eval()
del ck
print("params", sum(p.numel() for p in lm.parameters())/1e6, "M | vocab", tok.vocab_size,
      "| d_model", cfg.d_model, "| layers", cfg.n_layers, "| d_ff", cfg.d_ff, "| max_seq_len", cfg.max_seq_len,
      "| norm_first", cfg.norm_first, "| dropout", cfg.dropout)

<All keys matched successfully>
params 110.979072 M | vocab 10000 | d_model 1024 | layers 8 | d_ff 4096 | max_seq_len 2048 | norm_first False | dropout 0.1


## Cell B — [ANCILLARY] decoder features check

In [9]:
# =====================================================================================
# CELL B — per-token vectors from the c-1 decoder + sanity check
# [c-1] c-1 has no encoder: the features are the decoder's last hidden states (after ln_f),
#       with the LM head skipped. Each token only sees the tokens to its LEFT (causal).
# =====================================================================================
def features(idx):                        # idx: (B, T) LongTensor, T <= 2048
    x = lm.tok_emb(idx)
    if lm.cfg.pos_emb_type == "sinusoidal":
        x = x + lm.pos_emb[:idx.size(1), :]
    for blk in lm.blocks:
        x = blk(x)
    return lm.ln_f(x)                     # (B, T, 1024)

# Sanity check: the LM head on these features must give the full model's logits
idx = torch.randint(4, cfg.vocab_size, (1, 64), device=device)
with torch.no_grad():
    f = features(idx); diff = (lm.head(f) - lm(idx)).abs().max().item()
print("max diff", diff, tuple(f.shape))   # expect 0.0, (1, 64, 1024)

max diff 0.0 (1, 64, 1024)


## Cell C — [ANCILLARY] LSI data + statute set

In [10]:
# =====================================================================================
# CELL C — [ANCILLARY] Phase 5 (LSI) data: load train / dev / test cases and the statute set
# One case = one long judgment -> a SET of statute IDs (multi-label).
# Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import json
import numpy as np
from collections import Counter
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "Exploration-Lab/CS-781-Capstone"
P5_DIR = "Phase_5_LSI"
print("Phase 5 files on HF:", [f for f in list_repo_files(REPO, repo_type="dataset") if "Phase_5" in f])

def get5(name):
    """Reads Phase_5_LSI/<name>. Accepts plain JSON or JSON-lines."""
    p = hf_hub_download(REPO, f"{P5_DIR}/{name}", repo_type="dataset")
    with open(p, encoding="utf-8") as f:
        txt = f.read()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return [json.loads(line) for line in txt.splitlines() if line.strip()]

_raw = get5("Train/train.json")
_first = _raw[0] if isinstance(_raw, list) else next(iter(_raw.values()))
print("train layout:", type(_raw).__name__, len(_raw), "| first record keys:",
      list(_first) if isinstance(_first, dict) else type(_first).__name__)

TEXT_FIELDS  = ("text", "document", "case_text", "judgment", "sentences")
LABEL_FIELDS = ("labels", "label", "statutes", "statute_ids", "target", "y")

def _as_label_list(lab, k):
    """Any of [1, 7], "1,7", 7, None -> sorted list of unique ints."""
    if lab is None:
        return None
    if isinstance(lab, str):
        lab = [x for x in lab.replace(";", ",").split(",") if x.strip()]
    if not isinstance(lab, (list, tuple)):
        lab = [lab]
    out = sorted({int(x) for x in lab})
    return out

def as_cases(raw, name):
    """-> {case_id: {"text": str, "labels": [int, ...] or None}}, in file order.
    Accepts a list of dicts, {case_id: {...}}, a one-key wrapper like {"data": [...]},
    or columns {"id": [...], "text": [...], ("labels": [...])}.
    Repeated ids: exact duplicates dropped; differing repeats kept as "<id>#dup<n>" (train/dev),
    dropped with a warning on test (one prediction per id)."""
    while isinstance(raw, dict) and len(raw) == 1 and isinstance(next(iter(raw.values())), (list, dict)):
        raw = next(iter(raw.values()))
    if isinstance(raw, dict) and "id" in raw and isinstance(raw["id"], list):
        keys = list(raw)
        raw = [{k: raw[k][i] for k in keys} for i in range(len(raw["id"]))]
    items = [(d.get("id", n), d) for n, d in enumerate(raw)] if isinstance(raw, list) else list(raw.items())
    cases, n_exact, n_diff = {}, 0, 0
    for k, d in items:
        assert isinstance(d, dict), f"case {k}: unexpected entry {type(d).__name__}"
        k = str(d.get("id", k))
        tf = next((f for f in TEXT_FIELDS if f in d), None)
        assert tf is not None, f"case {k}: no text field (keys {list(d)})"
        text = d[tf]
        if isinstance(text, list):                    # a list of sentences/paragraphs
            text = " ".join(map(str, text))
        lf = next((f for f in LABEL_FIELDS if f in d), None)
        rec = {"text": str(text), "labels": _as_label_list(d[lf] if lf else None, k)}
        if k in cases:
            if cases[k] == rec:
                n_exact += 1
                continue
            if name == "test":
                print(f"WARNING test id {k} appears twice with different text: keeping the first")
                continue
            n_diff += 1
            j = 1
            while f"{k}#dup{j}" in cases:
                j += 1
            k = f"{k}#dup{j}"
        cases[k] = rec
    print(f"{name}: {len(items)} records -> {len(cases)} cases | exact duplicates dropped {n_exact} | "
          f"repeated ids with different text/labels kept {n_diff}")
    return cases

train = as_cases(_raw, "train")
dev   = as_cases(get5("Train/dev.json"), "dev")
test  = as_cases(get5("Test/test.json"), "test")
del _raw

assert all(d["labels"] for s in (train, dev) for d in s.values()), \
    "every train/dev case needs at least one statute: check LABEL_FIELDS"
_tl = sum(d["labels"] is not None for d in test.values())
if _tl:
    print(f"test has a label field on {_tl} cases (ignored, set to None)")
for d in test.values():
    d["labels"] = None

# -------------------------------------------------------------------------------------
# Statute set = every ID seen in train or dev. Model output j <-> statute STATUTES[j].
# -------------------------------------------------------------------------------------
STATUTES = sorted({l for s in (train, dev) for d in s.values() for l in d["labels"]})
S2J = {s: j for j, s in enumerate(STATUTES)}
N_LABELS = len(STATUTES)
VALID_STATUTES = set(STATUTES)

tr_cnt = Counter(l for d in train.values() for l in d["labels"])
dv_cnt = Counter(l for d in dev.values() for l in d["labels"])
print(f"\n{N_LABELS} statutes | ID range {STATUTES[0]}..{STATUTES[-1]} | "
      f"in train {len(tr_cnt)} | in dev {len(dv_cnt)} | dev-only (never in train) {sorted(set(dv_cnt) - set(tr_cnt))}")
for name, split in (("train", train), ("dev", dev)):
    n = np.array([len(d["labels"]) for d in split.values()])
    print(f"{name:5s}: {len(split):6d} cases | statutes per case min {n.min()} mean {n.mean():.2f} "
          f"median {np.median(n):.0f} max {n.max()}")
print(f"test : {len(test):6d} cases")

f = np.array([tr_cnt.get(s, 0) for s in STATUTES])
for lo, hi in ((0, 0), (1, 5), (6, 20), (21, 100), (101, 10**9)):
    print(f"  statutes with {lo}-{hi if hi < 10**9 else 'inf'} train cases: {int(((f >= lo) & (f <= hi)).sum())}")
print("most frequent:", tr_cnt.most_common(10))
print("rarest       :", sorted(tr_cnt.items(), key=lambda x: x[1])[:10])
print("first test ids:", list(test)[:3])
print(f"train∩dev ids {len(set(train) & set(dev))} | train∩test {len(set(train) & set(test))}")

Phase 5 files on HF: ['Phase_5_LSI/Test/test.json', 'Phase_5_LSI/Train/dev.json', 'Phase_5_LSI/Train/train.json']


Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

train layout: list 42750 | first record keys: ['id', 'text', 'labels']
train: 42750 records -> 42750 cases | exact duplicates dropped 0 | repeated ids with different text/labels kept 0


Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

dev: 10181 records -> 10181 cases | exact duplicates dropped 0 | repeated ids with different text/labels kept 0


Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

test: 13019 records -> 13019 cases | exact duplicates dropped 0 | repeated ids with different text/labels kept 0

100 statutes | ID range 0..99 | in train 100 | in dev 100 | dev-only (never in train) []
train:  42750 cases | statutes per case min 1 mean 3.70 median 3 max 31
dev  :  10181 cases | statutes per case min 1 mean 3.92 median 3 max 28
test :  13019 cases
  statutes with 0-0 train cases: 0
  statutes with 1-5 train cases: 0
  statutes with 6-20 train cases: 0
  statutes with 21-100 train cases: 0
  statutes with 101-inf train cases: 100
most frequent: [(5, 11730), (34, 10380), (97, 7880), (43, 7728), (39, 7120), (12, 4324), (13, 4136), (10, 4134), (77, 4046), (94, 3835)]
rarest       : [(84, 272), (25, 287), (23, 304), (92, 309), (66, 309), (59, 310), (50, 318), (74, 319), (27, 329), (15, 336)]
first test ids: ['100002997', '100004477', '1000086']
train∩dev ids 0 | train∩test 0


## Cell D — [ANCILLARY] tokenise, multi-label scorer, writer, baseline

In [11]:
# =====================================================================================
# CELL D — [ANCILLARY] document tokenisation, multi-label scorer, prediction writer, baseline
# Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import json, os, zipfile
import numpy as np
from collections import Counter
from tqdm.auto import tqdm

PAD_ID = SPECIAL_TOKENS.index("<pad>")   # 0
UNK_ID = SPECIAL_TOKENS.index("<unk>")   # 1

def encode_doc(text):
    ids = tok.encode(text)
    return ids if ids else [UNK_ID]

# Tokenise every case once: d["ids"] = token ids of the whole judgment (re-run if you change encode_doc)
for name, split in (("train", train), ("dev", dev), ("test", test)):
    for d in tqdm(split.values(), desc=f"tokenising {name}"):
        d["ids"] = encode_doc(d["text"])

# Length statistics in c-1 tokens, to choose WINDOW (<= 2048), STRIDE and MAX_WINDOWS in Cell E
for name, split in (("train", train), ("dev", dev), ("test", test)):
    L = np.array([len(d["ids"]) for d in split.values()])
    print(f"{name:5s} tokens: min {L.min()} median {np.median(L):.0f} p90 {np.percentile(L, 90):.0f} "
          f"p99 {np.percentile(L, 99):.0f} max {L.max()} | > 2048: {(L > 2048).mean():.1%}")


# -------------------------------------------------------------------------------------
# Local scorer. pred / gold are SETS of statute IDs per case.
#   macro-F1 (primary): mean per-statute F1 over the statutes in the REFERENCE (gold) labels of the
#                       scored split, the way the spec says the evaluator sets the label dimension.
#                       A statute that is never gold but is predicted only costs micro precision.
#   macro-F1 (all):     the same mean over every statute in STATUTES (stricter; shown for reference).
#   micro-F1, subset accuracy (exact set match): secondary metrics.
# -------------------------------------------------------------------------------------
def ml_scores(gold_sets, pred_sets):
    tp, fp, fn = Counter(), Counter(), Counter()
    exact = 0
    for g, p in zip(gold_sets, pred_sets):
        g, p = set(g), set(p)
        exact += g == p
        for s in g & p: tp[s] += 1
        for s in p - g: fp[s] += 1
        for s in g - p: fn[s] += 1
    def f1(s):
        P = tp[s] / (tp[s] + fp[s]) if tp[s] + fp[s] else 0.0
        R = tp[s] / (tp[s] + fn[s]) if tp[s] + fn[s] else 0.0
        return P, R, (2 * P * R / (P + R) if P + R else 0.0), tp[s] + fn[s]
    ref = sorted({s for g in gold_sets for s in g})
    per = {s: f1(s) for s in sorted(set(ref) | set(STATUTES))}
    TP, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    mP = TP / (TP + FP) if TP + FP else 0.0
    mR = TP / (TP + FN) if TP + FN else 0.0
    return {"f1": float(np.mean([per[s][2] for s in ref])),
            "f1_all": float(np.mean([per[s][2] for s in STATUTES])),
            "p": float(np.mean([per[s][0] for s in ref])), "r": float(np.mean([per[s][1] for s in ref])),
            "micro_f1": 2 * mP * mR / (mP + mR) if mP + mR else 0.0, "micro_p": mP, "micro_r": mR,
            "subset_acc": exact / max(1, len(gold_sets)), "per": per}

def evaluate(pred, docs, show=True):
    """pred: {case_id: [statute IDs]}. Scores every case in docs."""
    ids = list(docs)
    res = ml_scores([docs[k]["labels"] for k in ids], [pred[k] for k in ids])
    if show:
        n_pred = np.mean([len(pred[k]) for k in ids])
        print(f"macro-F1 {res['f1']:.4f} (P {res['p']:.4f} R {res['r']:.4f}) | macro-F1 over all statutes {res['f1_all']:.4f} | "
              f"micro-F1 {res['micro_f1']:.4f} (P {res['micro_p']:.4f} R {res['micro_r']:.4f}) | "
              f"subset acc {res['subset_acc']:.4f} | statutes predicted per case {n_pred:.2f}")
    return res

def per_statute_report(res, k=15):
    """Per-statute F1 with train frequency: the rarest and the worst statutes (for the report)."""
    rows = [(s, *res["per"][s], tr_cnt.get(s, 0)) for s in STATUTES]
    print(f"{'statute':>8s} {'P':>6s} {'R':>6s} {'F1':>6s} {'dev n':>6s} {'train n':>8s}")
    for title, key in (("rarest in train", lambda r: r[5]), ("lowest F1 (dev n > 0)", lambda r: (r[3], -r[4]))):
        print(f"-- {title}")
        for s, P, R, F, n, ntr in sorted([r for r in rows if r[4] > 0], key=key)[:k]:
            print(f"{s:8d} {P:6.3f} {R:6.3f} {F:6.3f} {n:6d} {ntr:8d}")
    for lo, hi in ((1, 5), (6, 20), (21, 100), (101, 10**9)):
        f = [r[3] for r in rows if r[4] > 0 and lo <= r[5] <= hi]
        if f:
            print(f"statutes with {lo}-{hi if hi < 10**9 else 'inf'} train cases: {len(f)} | mean dev F1 {np.mean(f):.4f}")

try:   # cross-check the macro / micro numbers against sklearn once
    from sklearn.metrics import f1_score
    from sklearn.preprocessing import MultiLabelBinarizer
    _g, _p = [[STATUTES[0]], [STATUTES[0], STATUTES[-1]], [STATUTES[-1]]], [[STATUTES[0]], [STATUTES[0]], []]
    _m = MultiLabelBinarizer(classes=sorted({s for g in _g for s in g})).fit(_g)
    _r = ml_scores(_g, _p)
    assert abs(_r["f1"] - f1_score(_m.transform(_g), _m.transform(_p), average="macro", zero_division=0)) < 1e-9
    assert abs(_r["micro_f1"] - f1_score(_m.transform(_g), _m.transform(_p), average="micro", zero_division=0)) < 1e-9
    print("scorer matches sklearn macro / micro F1")
except ImportError:
    pass


# -------------------------------------------------------------------------------------
# Writer: prediction.json = LIST of {"id", "pred": [int, ...]} in test order, zipped.
# Checks: one entry per test id, plain Python ints, no duplicates, every ID a known statute.
# -------------------------------------------------------------------------------------
def write_prediction(pred, test_docs, out_dir="/kaggle/working"):
    os.makedirs(out_dir, exist_ok=True)
    assert set(pred) >= set(test_docs), "missing test ids"
    out, n_empty = [], 0
    for case_id in test_docs:
        p = sorted({int(s) for s in pred[case_id]})      # numpy ints -> plain int, deduplicated
        bad = [s for s in p if s not in VALID_STATUTES]
        assert not bad, f"{case_id}: unknown statute IDs {bad}"
        n_empty += not p
        out.append({"id": case_id, "pred": p})
    assert len(out) == len(test_docs) and len({o["id"] for o in out}) == len(out), "one entry per test case"
    if n_empty:
        print(f"WARNING {n_empty} test cases have an empty pred list (the spec does not say if [] is accepted)")
    path = os.path.join(out_dir, "prediction.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False)
    back = json.load(open(path))
    assert isinstance(back, list) and all(set(o) == {"id", "pred"} and isinstance(o["pred"], list)
                                          and all(type(s) is int for s in o["pred"])
                                          and len(set(o["pred"])) == len(o["pred"]) for o in back)
    zpath = os.path.join(out_dir, "submission.zip")
    with zipfile.ZipFile(zpath, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    with zipfile.ZipFile(zpath) as z:
        assert z.namelist() == ["prediction.json"], z.namelist()
    n = [len(o["pred"]) for o in out]
    print(f"{len(out)} cases | statutes per case mean {np.mean(n):.2f} max {max(n)} | "
          f"distinct statutes predicted {len({s for o in out for s in o['pred']})} -> {zpath}")


# -------------------------------------------------------------------------------------
# Baseline: the K most frequent train statutes for every case (K = median statutes per train case).
# Checks the scorer and writer end to end before any modelling.
# -------------------------------------------------------------------------------------
_K = int(np.median([len(d["labels"]) for d in train.values()]))
_top = [s for s, _ in tr_cnt.most_common(_K)]
print(f"baseline: top-{_K} statutes {_top} for every case")
evaluate({k: _top for k in dev}, dev)
write_prediction({k: _top for k in test}, test, out_dir="/kaggle/working/baseline")   # format check

tokenising train:   0%|          | 0/42750 [00:00<?, ?it/s]

tokenising dev:   0%|          | 0/10181 [00:00<?, ?it/s]

tokenising test:   0%|          | 0/13019 [00:00<?, ?it/s]

train tokens: min 2 median 721 p90 4655 p99 14103 max 160535 | > 2048: 27.5%
dev   tokens: min 4 median 916 p90 5068 p99 15430 max 130642 | > 2048: 31.6%
test  tokens: min 2 median 905 p90 4813 p99 14919 max 258440 | > 2048: 30.3%
scorer matches sklearn macro / micro F1
baseline: top-3 statutes [5, 34, 97] for every case
macro-F1 0.0118 (P 0.0074 R 0.0300) | macro-F1 over all statutes 0.0118 | micro-F1 0.2131 (P 0.2456 R 0.1882) | subset acc 0.0002 | statutes predicted per case 3.00
13019 cases | statutes per case mean 3.00 max 3 | distinct statutes predicted 3 -> /kaggle/working/baseline/submission.zip


## Cell E0 — [ANCILLARY] c-1 decoder path, bias groups, windows, loss, thresholds, training loop

In [12]:
# =====================================================================================
# CELL E0 — [ANCILLARY] shared plumbing for the c-1 Phase 5 (LSI) experiments
# Run once after Cells 0, A, B, C, D. Then run Cell E (BitFit).
# Needs from earlier cells: lm, cfg, device, features, train, dev, test (with d["ids"]),
#   STATUTES, S2J, N_LABELS, tr_cnt, evaluate, write_prediction
#
# Same as the c-6 Phase 5 E0 (windows, multi-hot BCE, pos_weight, aggregation, thresholds,
# training loop with checkpoints/resume), with the c-6 encoder path swapped for the c-1
# DECODER path. Lines marked [c-1] are where this differs from the c-6 version.
#
# Windows: full-length windows of WINDOW tokens every STRIDE tokens, the last one ending at
# the document end; a short case is one window of its own length. Batches hold windows of
# IDENTICAL length, so nothing is ever padded. Every window of a case is trained with that
# case's full statute set (multi-hot target); at prediction time the case's window
# probabilities are combined into one probability per statute, then thresholded.
# Nothing in this cell is a model design or a hyperparameter choice: every choice
# (window, aggregation, threshold, weighting) is an argument you set in Cell E.
# =====================================================================================
import math, time, random
import numpy as np
from collections import defaultdict, Counter
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm

FEAT_DIM = cfg.d_model   # [c-1] 1024 (c-6 was 560)
# [c-1] nothing to drop: c-1 is decoder-only. lm.head (tied to tok_emb) is simply never called.


# -------------------------------------------------------------------------------------
# [c-1] decoder path (identical to Cell B features(), but on any backbone object)
# -------------------------------------------------------------------------------------
# FAST_ATTENTION: same maths as CausalSelfAttention.forward (same weights, same causal mask,
# same attention_scale), but softmax(QK^T * scale + causal mask) V runs in PyTorch's fused
# scaled_dot_product_attention instead of building the (B, 8, T, T) score matrix by hand.
# d_head = 1024 / 8 = 128, so the memory-efficient kernel can be used. Checked and timed at
# the end of this cell; switched off automatically if it is not faster or not equal.
FAST_ATTENTION = True

def _self_attn_fast(attn, x):
    B, T, C = x.shape
    q, k, v = [p(x).view(B, T, attn.n_heads, attn.d_head).transpose(1, 2)
               for p in (attn.q_proj, attn.k_proj, attn.v_proj)]
    out = F.scaled_dot_product_attention(q, k, v, dropout_p=attn.attn_dropout.p if attn.training else 0.0,
                                         is_causal=True, scale=attn.attention_scale)
    out = out.transpose(1, 2).contiguous().view(B, T, C)
    return attn.resid_dropout(attn.out_proj(out))


# GRAD_CHECKPOINT: memory saver for long windows, same maths. During training each block's
# inner activations are not kept for backward; the block is recomputed in the backward pass
# instead (~30% slower, far less GPU memory). Set it in Cell E.
GRAD_CHECKPOINT = False

def _block(blk, x):
    if not FAST_ATTENTION:
        return blk(x)
    if blk.norm_first:                          # Block.forward (pre-norm) with the fused attention
        x = x + _self_attn_fast(blk.attn, blk.ln1(x))
        return x + blk.ffn(blk.ln2(x))
    x = blk.ln1(x + _self_attn_fast(blk.attn, x))   # Block.forward (post-norm)
    return blk.ln2(x + blk.ffn(x))


def decoder_forward(backbone, idx):
    T = idx.size(1)
    assert T <= backbone.cfg.max_seq_len, f"window longer than {backbone.cfg.max_seq_len}"
    x = backbone.tok_emb(idx)
    if backbone.cfg.pos_emb_type == "sinusoidal":
        x = x + backbone.pos_emb[:T, :]
    use_ckpt = GRAD_CHECKPOINT and torch.is_grad_enabled() and backbone.training
    for blk in backbone.blocks:                 # 8 causal decoder Blocks
        x = checkpoint(_block, blk, x, use_reentrant=False) if use_ckpt else _block(blk, x)
    return backbone.ln_f(x)                     # (B, T, 1024); LM head skipped


# -------------------------------------------------------------------------------------
# [c-1] Bias terms of the decoder, by group (for BitFit in Cell E)
#   ffn_in  : ffn.net[0].bias  (1024 -> d_ff), one per block
#   ffn_out : ffn.net[2].bias  (d_ff -> 1024), one per block
#   ln1, ln2: LayerNorm biases, one each per block
#   ln_f    : final LayerNorm bias
#   The attention q/k/v/out projections were built with bias=False: there is no attention bias.
# -------------------------------------------------------------------------------------
BIAS_GROUPS = ("ffn_in", "ffn_out", "ln1", "ln2", "ln_f")

def bias_group(name):
    if ".ffn.net.0." in name: return "ffn_in"
    if ".ffn.net.2." in name: return "ffn_out"
    if ".ln1." in name: return "ln1"
    if ".ln2." in name: return "ln2"
    if name.startswith("ln_f."): return "ln_f"
    return None

def decoder_biases(backbone):
    """{param name: (group, tensor)} for every bias of the c-1 decoder."""
    out = {}
    for n, p in backbone.named_parameters():
        if n.endswith(".bias"):
            g = bias_group(n)
            assert g is not None, f"unexpected bias {n}"
            out[n] = (g, p)
    return out


class LSIModel(nn.Module):
    """backbone (c-1 decoder path) -> head -> N_LABELS logits per WINDOW (one per statute).
    head: YOUR module, (B, T, 1024) token vectors of B windows -> (B, N_LABELS) logits."""
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone, self.head = backbone, head
    def forward(self, idx):
        return self.head(decoder_forward(self.backbone, idx))   # (B, N_LABELS)


def identity_probe():
    """64 real tokens from a training case, for the checks."""
    d = next(iter(train.values()))
    return torch.tensor([d["ids"][:64]], device=device)


def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / total {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot


# -------------------------------------------------------------------------------------
# Windows (Phase 2 make_windows, unchanged) + which windows stand for the case
#   DOC_REPR "all" = every window. Judgments are long, so you can also keep at most
#   max_windows of them: "first", "last", "first_last" (half from each end) or "spread"
#   (evenly over the whole case). Windows stay in document order.
# -------------------------------------------------------------------------------------
DOC_REPRS = ("all", "first", "last", "first_last", "spread")

def make_windows(n, window, stride):
    if n <= window:
        return [(0, n)]
    starts = list(range(0, n - window, stride)) + [n - window]
    return [(s, s + window) for s in starts]


def select_windows(wins, doc_repr, max_windows):
    n = len(wins)
    if doc_repr == "all" or n <= max_windows:
        return wins
    if doc_repr == "first":
        return wins[:max_windows]
    if doc_repr == "last":
        return wins[-max_windows:]
    if doc_repr == "first_last":
        h = (max_windows + 1) // 2
        return wins[:h] + wins[n - (max_windows - h):] if max_windows > h else wins[:h]
    if doc_repr == "spread":
        idx = sorted({round(i * (n - 1) / max(1, max_windows - 1)) for i in range(max_windows)})
        return [wins[i] for i in idx]
    raise ValueError(doc_repr)


def build_examples(docs, window, stride, doc_repr="all", max_windows=None):
    """One example per window: {"doc", "w", "ids", "labels"} (labels = the case's statute set, None for test)."""
    assert window <= cfg.max_seq_len and doc_repr in DOC_REPRS          # [c-1] 2048
    assert doc_repr == "all" or max_windows, "set max_windows for this doc_repr"
    examples = []
    for doc_id, d in tqdm(docs.items(), desc="windowing", leave=False):
        ids = d["ids"]
        wins = select_windows(make_windows(len(ids), window, stride), doc_repr, max_windows)
        for w, (s, e) in enumerate(wins):
            examples.append({"doc": doc_id, "w": w, "ids": ids[s:e], "labels": d["labels"]})
    per = Counter(ex["doc"] for ex in examples)
    print(f"{len(per)} cases -> {len(examples)} windows | windows per case: mean {len(examples)/len(per):.2f} max {max(per.values())}")
    return examples


def make_batches(examples, batch_size, shuffle):
    """Batches of windows with IDENTICAL length (no padding needed)."""
    by_len = defaultdict(list)
    for ex in examples:
        by_len[len(ex["ids"])].append(ex)
    out = []
    for exs in by_len.values():
        if shuffle:
            random.shuffle(exs)
        out += [exs[i:i + batch_size] for i in range(0, len(exs), batch_size)]
    if shuffle:
        random.shuffle(out)
    return out


def multi_hot(statutes):
    v = torch.zeros(N_LABELS)
    v[[S2J[s] for s in statutes]] = 1.0
    return v


def to_tensors(batch):
    b = {"ids": torch.tensor([ex["ids"] for ex in batch], device=device)}
    if batch[0]["labels"] is not None:
        b["labels"] = torch.stack([multi_hot(ex["labels"]) for ex in batch]).to(device)   # (B, N_LABELS) 0/1
    return b


# -------------------------------------------------------------------------------------
# Loss: one independent sigmoid per statute (binary relevance), BCE against the multi-hot target.
#   pos_weight (N_LABELS,): multiplies the loss on POSITIVE targets of each statute, the usual
#   PyTorch way to push recall up on rare statutes. make_pos_weight builds it from train counts:
#     "none"         -> no weighting
#     "neg_pos"      -> (#cases without the statute) / (#cases with it)
#     "sqrt_neg_pos" -> square root of that (milder)
#   cap: optional upper limit on any weight (very rare statutes get huge ratios). Your choice.
# -------------------------------------------------------------------------------------
def bce_loss(logits, targets, pos_weight=None):
    return F.binary_cross_entropy_with_logits(logits.float(), targets, pos_weight=pos_weight)


def make_pos_weight(scheme, cap=None):
    assert scheme in ("none", "neg_pos", "sqrt_neg_pos"), "set POS_WEIGHT_SCHEME"
    if scheme == "none":
        return None
    n = len(train)
    pos = torch.tensor([max(1, tr_cnt.get(s, 0)) for s in STATUTES], dtype=torch.float)
    w = (n - pos) / pos
    if scheme == "sqrt_neg_pos":
        w = w.sqrt()
    if cap is not None:
        w = w.clamp(max=cap)
    print(f"pos_weight: min {w.min():.2f} median {w.median():.2f} max {w.max():.2f}")
    return w.to(device)


# -------------------------------------------------------------------------------------
# Prediction: per-window sigmoid probabilities -> one probability vector per case -> statute set
#   agg (yours): "mean" = average over the case's windows, "max" = largest window probability
#   threshold (yours): one float for every statute (global) or an array of N_LABELS (per statute)
#   min_labels (yours): if fewer statutes pass the threshold, take the min_labels most probable
#                       ones instead (0 = allow an empty set; the spec does not say if [] is accepted)
# -------------------------------------------------------------------------------------
AGGS = ("mean", "max")

@torch.no_grad()
def predict_probs(model, examples, batch_size, agg):
    """-> {case_id: np.array (N_LABELS,)} aggregated window probabilities."""
    assert agg in AGGS, agg
    model.eval()
    per_doc = defaultdict(list)
    for batch in tqdm(make_batches(examples, batch_size, shuffle=False), desc="predict", leave=False):
        with torch.autocast("cuda", dtype=torch.float16):
            pr = model(to_tensors(batch)["ids"]).float().sigmoid().cpu().numpy()
        for ex, p in zip(batch, pr):
            per_doc[ex["doc"]].append(p)
    if agg == "mean":
        return {d: np.mean(v, axis=0) for d, v in per_doc.items()}
    return {d: np.max(v, axis=0) for d, v in per_doc.items()}


def decide(probs, threshold, min_labels=0):
    """probs {case_id: (N_LABELS,)} -> {case_id: [statute IDs]}."""
    thr = np.broadcast_to(np.asarray(threshold, dtype=float), (N_LABELS,))
    out = {}
    for d, p in probs.items():
        js = np.flatnonzero(p >= thr)
        if len(js) < min_labels:
            js = np.argsort(-p)[:min_labels]
        out[d] = [STATUTES[j] for j in js]
    return out


def predict(model, examples, batch_size, agg, threshold, min_labels=0):
    return decide(predict_probs(model, examples, batch_size, agg), threshold, min_labels)


# -------------------------------------------------------------------------------------
# Threshold selection on dev (the spec allows dev for this). grid = the candidate values (yours).
#   "global":    one threshold for every statute, the grid value with the best dev macro-F1
#   "per_label": start from the best global value, then for each statute separately the grid
#                value with the best F1 on that statute (statutes with no dev positives keep the
#                global value). Per-label thresholds fit dev closely: rare statutes have very few
#                dev cases, so expect dev macro-F1 to overstate the test score.
# -------------------------------------------------------------------------------------
def tune_thresholds(probs, docs, mode, grid, min_labels=0):
    assert mode in ("global", "per_label") and grid is not None and len(grid), "set THRESH_MODE / THRESH_GRID"
    ids = list(docs)
    P = np.stack([probs[k] for k in ids])                                   # (n_cases, N_LABELS)
    Y = np.stack([multi_hot(docs[k]["labels"]).numpy() for k in ids]).astype(bool)
    scores = [(evaluate(decide(probs, t, min_labels), docs, show=False)["f1"], t) for t in grid]
    best_f1, best_t = max(scores)
    print("global:", " ".join(f"{t:.2f}->{f:.4f}" for f, t in scores), f"| best {best_t} (dev macro-F1 {best_f1:.4f})")
    if mode == "global":
        return float(best_t)
    thr = np.full(N_LABELS, float(best_t))
    for j in range(N_LABELS):
        if not Y[:, j].any():
            continue
        best = (-1.0, best_t)
        for t in grid:
            pj = P[:, j] >= t
            tp = (pj & Y[:, j]).sum(); fp = (pj & ~Y[:, j]).sum(); fn = (~pj & Y[:, j]).sum()
            f = 2 * tp / (2 * tp + fp + fn) if tp else 0.0
            best = max(best, (f, t))
        thr[j] = best[1]
    f = evaluate(decide(probs, thr, min_labels), docs, show=False)["f1"]
    print(f"per_label: dev macro-F1 {f:.4f} | thresholds min {thr.min():.2f} median {np.median(thr):.2f} max {thr.max():.2f}")
    return thr


# -------------------------------------------------------------------------------------
# Training loop (the c-6 Phase 5 train_model). loss_fn(logits, batch_tensors) comes from the
# technique cell. Keeps the epoch with the best dev macro-F1 (case level, fixed threshold /
# min_labels). lr_lambda: optional step -> multiplier (your schedule), or None for constant LR.
# param_groups: [{"params": [...], "lr": ...}, ...] so biases and head can use different LRs
#   (None = every trainable parameter at lr). [c-1] added for BitFit; same as Phase 4 c-1.
# -------------------------------------------------------------------------------------
def train_model(model, train_ex, val_ex, val_docs, loss_fn, *, lr, epochs, batch_size, agg, threshold,
                min_labels=0, weight_decay=0.0, grad_clip=1.0, lr_lambda=None, save_path="best_trainable.pt",
                max_epoch_minutes=None, micro_batch=None, last_path=None, resume_from=None, param_groups=None):
    """max_epoch_minutes: stop each epoch's training after this many minutes (batches are shuffled,
    so the epoch then covers a random part of the training windows). Dev scoring still runs.
    micro_batch: memory saver (gradient accumulation). Each batch of batch_size windows is run
    through the model micro_batch windows at a time and the gradients are summed before ONE
    optimizer step, so the update is the same as with the full batch. None = whole batch at once.
    last_path: after EVERY epoch, save a full checkpoint there (trainable weights of that epoch,
    the best epoch's weights, optimizer / scaler / schedule state, epoch number, best dev F1, log).
    resume_from: path to such a checkpoint (e.g. from an earlier notebook version attached as
    input). Training then continues at the next epoch and runs UP TO `epochs` in total, so
    epochs=6 after a 3-epoch run trains epochs 4, 5 and 6. The best weights so far are written
    to save_path first, so they are kept even if no new epoch beats them."""
    params = [p for p in model.parameters() if p.requires_grad]
    if param_groups is None:
        param_groups = [{"params": params, "lr": lr}]
    assert {id(p) for g in param_groups for p in g["params"]} == {id(p) for p in params}, \
        "param_groups must cover every trainable parameter exactly"
    opt = torch.optim.AdamW(param_groups, lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda) if lr_lambda else None
    scaler = torch.amp.GradScaler("cuda")
    torch.cuda.reset_peak_memory_stats()
    t0, best_f1, best_res, log = time.time(), -1.0, None, []
    skipped = 0   # batches whose loss was NaN/inf (skipped, not trained on)
    start, best_state = 0, None
    if resume_from:
        ck = torch.load(resume_from, map_location="cpu", weights_only=False)
        r = model.load_state_dict(ck["trainable"], strict=False)
        names = {n for n, p in model.named_parameters() if p.requires_grad}
        assert not r.unexpected_keys and names <= set(ck["trainable"]), \
            f"checkpoint does not match this model (same BITFIT_GROUPS / head?) " \
            f"unexpected {r.unexpected_keys[:5]} missing {sorted(names - set(ck['trainable']))[:5]}"
        opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        if sched and ck.get("sched"): sched.load_state_dict(ck["sched"])
        start, best_f1, best_res, log = ck["epoch"], ck["best_f1"], ck["best_res"], ck["log"]
        skipped, best_state = ck.get("skipped", 0), ck["best"]
        torch.save(best_state, save_path)
        print(f"resumed from {resume_from}: {start} epochs done, best dev macro-F1 so far {best_f1:.4f}; "
              f"training epochs {start + 1}..{epochs}")
        assert start < epochs, f"checkpoint already has {start} epochs: set EPOCHS higher to train more"
    for epoch in range(start, epochs):
        model.train(); tot, nb = 0.0, 0
        batches = make_batches(train_ex, batch_size, shuffle=True)
        pbar = tqdm(batches, desc=f"epoch {epoch + 1}/{epochs}")
        t_ep, seen = time.time(), 0
        for batch in pbar:
            if max_epoch_minutes and time.time() - t_ep > 60 * max_epoch_minutes:
                tqdm.write(f"epoch {epoch + 1}: time cap {max_epoch_minutes} min reached after "
                           f"{seen}/{len(batches)} batches ({100 * seen / len(batches):.1f}% of the training windows)")
                break
            seen += 1
            opt.zero_grad(set_to_none=True)
            mb = micro_batch or len(batch)
            loss, bad = 0.0, False
            for i in range(0, len(batch), mb):
                part = batch[i:i + mb]
                b = to_tensors(part)
                with torch.autocast("cuda", dtype=torch.float16):
                    logits = model(b["ids"])
                l = loss_fn(logits.float(), b) * (len(part) / len(batch))   # weighted like the full-batch mean
                if not torch.isfinite(l):
                    bad = True
                    break
                scaler.scale(l).backward()
                loss += l.detach()
            if bad:
                opt.zero_grad(set_to_none=True)
                skipped += 1
                continue
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, grad_clip)
            scaler.step(opt); scaler.update()
            if sched: sched.step()
            tot += loss.item(); nb += 1
            pbar.set_postfix(loss=f"{tot / nb:.4f}", skipped=skipped)
        res = evaluate(predict(model, val_ex, batch_size, agg, threshold, min_labels), val_docs, show=False)
        log.append({"epoch": epoch + 1, "train_loss": tot / max(1, nb), "batches": seen, "of": len(batches),
                    "dev_macro_f1": res["f1"], "dev_micro_f1": res["micro_f1"],
                    "dev_subset_acc": res["subset_acc"], "skipped": skipped})
        print(log[-1])
        cur = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
        if res["f1"] > best_f1:
            best_f1, best_res, best_state = res["f1"], {k: v for k, v in res.items() if k != "per"}, cur
            torch.save(cur, save_path)
        if last_path:
            torch.save({"trainable": cur, "best": best_state, "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "sched": sched.state_dict() if sched else None, "epoch": epoch + 1, "best_f1": best_f1,
                        "best_res": best_res, "log": log, "skipped": skipped}, last_path)
            tqdm.write(f"epoch {epoch + 1} checkpoint -> {last_path}")
    print(f"best dev macro-F1 {best_f1:.4f} (micro-F1 {best_res['micro_f1']:.4f}, subset acc {best_res['subset_acc']:.4f}) | "
          f"train time {(time.time() - t0) / 60:.1f} min | peak GPU mem {torch.cuda.max_memory_allocated() / 2**30:.2f} GB")
    return log, best_res


@torch.no_grad()
def time_inference(model, examples, batch_size, agg):
    """Wall-clock seconds to predict a set (for the efficiency table in the report)."""
    torch.cuda.synchronize(); t = time.time()
    predict_probs(model, examples, batch_size, agg)
    torch.cuda.synchronize(); return time.time() - t


def error_cases(pred, docs, k=5, chars=300):
    """Statutes most often missed / wrongly added on dev, and a few cases with both kinds of error."""
    missed, added = Counter(), Counter()
    for i, d in docs.items():
        g, p = set(d["labels"]), set(pred[i])
        missed.update(g - p); added.update(p - g)
    print("most often MISSED (statute: times):", missed.most_common(15))
    print("most often WRONGLY ADDED (statute: times):", added.most_common(15))
    bad = sorted(docs, key=lambda i: -len(set(docs[i]["labels"]) ^ set(pred[i])))[:k]
    for i in bad:
        g, p = set(docs[i]["labels"]), set(pred[i])
        t = docs[i]["text"]
        print(f"- {i} | {len(docs[i]['ids'])} tokens | gold {sorted(g)} | pred {sorted(p)} | "
              f"missed {sorted(g - p)} added {sorted(p - g)}\n  start: {t[:chars]!r}")


def write_submission(model, build_kwargs, batch_size, agg, threshold, min_labels=0, best_path=None,
                     out_dir="/kaggle/working"):
    """Reload the best trainable weights, predict test, write prediction.json + submission.zip (Cell D writer)."""
    if best_path:
        model.load_state_dict(torch.load(best_path), strict=False)
    test_ex = build_examples(test, **build_kwargs)
    write_prediction(predict(model, test_ex, batch_size, agg, threshold, min_labels), test, out_dir=out_dir)


# -------------------------------------------------------------------------------------
# [c-1] Checks: decoder_forward equals Cell B features(); FAST_ATTENTION equal + timed on one
# batch of 4 windows x 1024 tokens (forward + backward)
# -------------------------------------------------------------------------------------
def _time_decoder(fast, reps=3):
    global FAST_ATTENTION
    FAST_ATTENTION = fast
    idx = torch.randint(4, cfg.vocab_size, (4, min(1024, cfg.max_seq_len)), device=device)   # small batch: timing only
    emb = lm.tok_emb.weight
    was = emb.requires_grad
    emb.requires_grad_(True)                    # makes sure backward has a path even if lm is frozen
    torch.cuda.synchronize(); t = time.time()
    for _ in range(reps):
        with torch.autocast("cuda", dtype=torch.float16):
            decoder_forward(lm, idx).float().mean().backward()
    torch.cuda.synchronize()
    emb.requires_grad_(was)
    for p in lm.parameters():
        p.grad = None                           # leave no gradients behind
    return (time.time() - t) / reps

lm.eval()
with torch.no_grad():
    _p = identity_probe()
    _ref = features(_p)
    FAST_ATTENTION = False; _s = decoder_forward(lm, _p)
    FAST_ATTENTION = True;  _f = decoder_forward(lm, _p)
_d_plain = (_s - _ref).abs().max().item()
_diff = (_f - _s).abs().max().item()
print(f"decoder check: decoder_forward vs features() {_d_plain:.2e} (should be 0)")
assert _d_plain < 1e-4, "decoder_forward differs from Cell B features()"
try:
    _t_slow, _t_fast = _time_decoder(False), _time_decoder(True)
except torch.cuda.OutOfMemoryError:            # GPU still holds an earlier run: judge by equality only
    for p in lm.parameters():
        p.grad = None
    torch.cuda.empty_cache()
    _t_slow, _t_fast = float("nan"), float("nan")
    print("timing skipped (GPU memory busy): restart the session or free the previous run's memory")
FAST_ATTENTION = _diff < 1e-3 and not (_t_fast >= _t_slow)
print(f"fused attention: max diff {_diff:.2e} | step time plain {_t_slow:.3f}s vs fused {_t_fast:.3f}s "
      f"-> FAST_ATTENTION = {FAST_ATTENTION}")
_b = decoder_biases(lm)
print("decoder bias values by group:", {g: sum(p.numel() for gg, p in _b.values() if gg == g) for g in BIAS_GROUPS})
torch.cuda.empty_cache()

decoder check: decoder_forward vs features() 0.00e+00 (should be 0)
fused attention: max diff 1.43e-06 | step time plain 0.358s vs fused 0.158s -> FAST_ATTENTION = True
decoder bias values by group: {'ffn_in': 32768, 'ffn_out': 8192, 'ln1': 8192, 'ln2': 8192, 'ln_f': 1024}


## Cell E — BitFit (YOURS: fill the TODOs)

In [13]:

# =====================================================================================
# CELL E — BitFit on the c-1 decoder for LSI  <<< YOUR CELL >>>   (ref [8] Ben Zaken et al. 2022)
#
# Run after Cells 0, A, B, C, D and E0. To run again from scratch, re-run Cell A and E0 first
# (fresh `lm`, nothing left trainable from an earlier run).
#
# BitFit = freeze every weight matrix of the Phase 1 model and train only (some of) its bias
# terms, plus the new head. E0 lists the c-1 decoder's biases by group:
#   ffn_in / ffn_out  (the two FFN linears, per block)   ln1 / ln2 (per block)   ln_f (final)
#   The attention projections have bias=False: c-1 has NO attention biases.
#
# Already done for you in E0 (ancillary): tokens -> 8 causal decoder Blocks -> ln_f ->
# (B, T, 1024) -> your head. The LM head is never called.
#
# YOURS (marked TODO):  [H] every value   [S] which bias groups train   [N] the head
# Sections marked [ANCILLARY] are plumbing; nothing to change there.
# =====================================================================================

# -------------------------------------------------------------------------------------
# [H] Hyperparameters and design choices — TODO: all yours. None = not set; the assert stops you.
# -------------------------------------------------------------------------------------
WINDOW        = 1024   # tokens per window (<= 2048, c-1 max_seq_len)
STRIDE        = 512    # step between window starts (<= WINDOW)
DOC_REPR      = "all"  # long-document representation: "all", "first", "last", "first_last" or "spread"
MAX_WINDOWS   = None   # only needed when DOC_REPR != "all"

BITFIT_GROUPS = ("ffn_in", "ffn_out", "ln1", "ln2", "ln_f")   # [S] bias groups to train
LR            = 1e-4   # learning rate for the bias terms
HEAD_LR       = 3e-4   # learning rate for the head; None = same as LR
EPOCHS        = 1
MAX_EPOCH_MINUTES = None   # optional time cap per epoch (None = no cap)
BATCH_SIZE    = 4
WEIGHT_DECAY  = 0.0
LR_LAMBDA     = None   # optional: step -> LR multiplier, None = constant LR

# [ANCILLARY] GPU memory savers: same maths and same updates, only slower. Use them when a
# batch of long windows runs out of memory (CUDA OutOfMemoryError).
GRAD_CHECKPOINT = False    # True = recompute each decoder block in backward instead of storing it
MICRO_BATCH     = 4        # e.g. 4 = run each batch 4 windows at a time, one optimizer step per batch

# [ANCILLARY] continue an earlier run: path to its last_checkpoint.pt (attach that notebook's
# output as input; find the path with  !find /kaggle/input -name last_checkpoint.pt ).
# EPOCHS is the TOTAL: after a 3-epoch run, EPOCHS = 6 trains epochs 4-6. None = fresh start.
RESUME_FROM     = None

POS_WEIGHT_SCHEME = "sqrt_neg_pos"   # "none", "neg_pos" or "sqrt_neg_pos" (see make_pos_weight in E0)
POS_WEIGHT_CAP    = 10.0   # optional upper limit on any statute's weight

AGG           = "mean"   # window probabilities -> case: "mean" or "max"
THRESHOLD     = 0.5      # fixed threshold used during training to pick the best epoch
MIN_LABELS    = 0        # at least this many statutes per case (0 = empty set allowed)
THRESH_MODE   = "global" # after training, tuned on dev: "global", "per_label", or "fixed" (keep THRESHOLD)
THRESH_GRID   = [0.10, 0.15, 0.20, 0.25, 0.30,
                 0.35, 0.40, 0.45, 0.50, 0.55,
                 0.60, 0.65, 0.70, 0.75, 0.80]

SEED          = 42
for _n in ["WINDOW", "STRIDE", "DOC_REPR", "BITFIT_GROUPS", "LR", "EPOCHS", "BATCH_SIZE",
           "WEIGHT_DECAY", "POS_WEIGHT_SCHEME", "AGG", "THRESHOLD", "MIN_LABELS", "THRESH_MODE"]:
    assert globals()[_n] is not None, f"set {_n}"
assert 0 < STRIDE <= WINDOW <= cfg.max_seq_len, "need 0 < STRIDE <= WINDOW <= 2048"
assert DOC_REPR in DOC_REPRS, DOC_REPR
assert DOC_REPR == "all" or MAX_WINDOWS, "set MAX_WINDOWS"
assert THRESH_MODE == "fixed" or THRESH_GRID, "set THRESH_GRID"
assert set(BITFIT_GROUPS) <= set(BIAS_GROUPS) and BITFIT_GROUPS, f"BITFIT_GROUPS must be a non-empty subset of {BIAS_GROUPS}"


# -------------------------------------------------------------------------------------
# [ANCILLARY] run setup
# -------------------------------------------------------------------------------------
BUILD = dict(window=WINDOW, stride=STRIDE, doc_repr=DOC_REPR, max_windows=MAX_WINDOWS)
random.seed(SEED); torch.manual_seed(SEED)
RUN_NAME = f"c1_bitfit_lsi_{DOC_REPR}"
OUT_DIR = f"/kaggle/working/{RUN_NAME}"; os.makedirs(OUT_DIR, exist_ok=True)
SAVE = f"{OUT_DIR}/best_trainable.pt"
LAST = f"{OUT_DIR}/last_checkpoint.pt"
POS_WEIGHT = make_pos_weight(POS_WEIGHT_SCHEME, POS_WEIGHT_CAP)


# -------------------------------------------------------------------------------------
# [ANCILLARY] freeze all of c-1, then make the bias groups in BITFIT_GROUPS trainable
# -------------------------------------------------------------------------------------
lm.eval()
probe = identity_probe()
with torch.no_grad():
    ref = features(probe).float()

for p in lm.parameters():
    p.requires_grad = False

dec_bias = decoder_biases(lm)
for g in BIAS_GROUPS:
    _ps = [p for gg, p in dec_bias.values() if gg == g]
    print(f"  {g:8s} {len(_ps):2d} tensors {sum(p.numel() for p in _ps):8,d} values  "
          f"{'TRAINABLE' if g in BITFIT_GROUPS else 'frozen'}")
for n, (g, p) in dec_bias.items():
    p.requires_grad = g in BITFIT_GROUPS

bias_params = [p for g, p in dec_bias.values() if p.requires_grad]
n_bias_tr = sum(p.numel() for p in bias_params)
n_other_tr = sum(p.numel() for n, p in lm.named_parameters() if p.requires_grad and n not in dec_bias)
print(f"trainable decoder biases: {n_bias_tr:,} | other trainable decoder params: {n_other_tr:,}")
assert n_bias_tr > 0, "no bias is trainable"
assert n_other_tr == 0, "BitFit trains biases only; a non-bias decoder parameter is trainable"

with torch.no_grad():   # unfreezing changes nothing yet: the model must still give Cell B's features
    diff = (features(probe).float() - ref).abs().max().item()
print(f"after freezing: max |diff| vs Cell B = {diff:.2e} (should be 0)")
assert diff < 1e-4


# -------------------------------------------------------------------------------------
# [N] Head — TODO: yours (multi-label: one logit per statute, no softmax, BCE is applied in the loss)
#     input  h      : (B, T, 1024)       decoder vectors of B windows (never padded, every token is real).
#                                        c-1 is causal: token t has seen tokens 0..t of its window only.
#     output logits : (B, n_labels)      n_labels = N_LABELS statutes (column j = statute STATUTES[j])
#   Your Phase 4 c-1 head (mean pool + Dropout + Linear) and your Phase 5 c-6 LSIHead are
#   in phase4/c1_bitfit/cellE_bitfit_c1.py and phase5/c6/cellE2_lorma_c6.py if you want to
#   start from one of them; in_dim here is 1024, not 560.
# -------------------------------------------------------------------------------------
class LSIHead(nn.Module):
    def __init__(self, in_dim, n_labels):
        super().__init__()
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(in_dim, n_labels)

    def forward(self, h):
        # h: (B, T, in_dim)
        pooled = h.mean(dim=1)   # (B, in_dim)
        pooled = self.dropout(pooled)
        return self.fc(pooled)   # (B, n_labels), raw logits

head = LSIHead(FEAT_DIM, N_LABELS)
model = LSIModel(lm, head).to(device)
tr_params, tot_params = count_params(model)
with torch.no_grad():
    assert model(probe).shape == (1, N_LABELS), "head must return (B, N_LABELS)"

head_params = [p for p in model.head.parameters() if p.requires_grad]
param_groups = [{"params": bias_params, "lr": LR},
                {"params": head_params, "lr": LR if HEAD_LR is None else HEAD_LR}]


# -------------------------------------------------------------------------------------
# [ANCILLARY] data, training (best epoch by dev macro-F1 at THRESHOLD), threshold tuning, scoring
# -------------------------------------------------------------------------------------
train_ex = build_examples(train, **BUILD)
val_ex = build_examples(dev, **BUILD)

log, best_res = train_model(
    model, train_ex, val_ex, dev,
    loss_fn=lambda logits, b: bce_loss(logits, b["labels"], POS_WEIGHT),
    lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY,
    lr_lambda=LR_LAMBDA, save_path=SAVE, agg=AGG, threshold=THRESHOLD, min_labels=MIN_LABELS,
    max_epoch_minutes=MAX_EPOCH_MINUTES, micro_batch=MICRO_BATCH,
    last_path=LAST, resume_from=RESUME_FROM, param_groups=param_groups)
train_mem = torch.cuda.max_memory_allocated() / 2**30

model.load_state_dict(torch.load(SAVE), strict=False)
dev_probs = predict_probs(model, val_ex, BATCH_SIZE, AGG)
print(f"\n-- dev at the fixed THRESHOLD {THRESHOLD}")
evaluate(decide(dev_probs, THRESHOLD, MIN_LABELS), dev)
FINAL_THR = THRESHOLD if THRESH_MODE == "fixed" else tune_thresholds(dev_probs, dev, THRESH_MODE, THRESH_GRID, MIN_LABELS)
dev_pred = decide(dev_probs, FINAL_THR, MIN_LABELS)
print(f"\n-- dev with the final thresholds ({THRESH_MODE})")
res = evaluate(dev_pred, dev)
per_statute_report(res)
error_cases(dev_pred, dev)
inf_s = time_inference(model, val_ex, BATCH_SIZE, AGG)
print(f"\n{RUN_NAME} | BitFit groups {BITFIT_GROUPS} | trainable {tr_params:,} / {tot_params:,} "
      f"({n_bias_tr:,} biases + {tr_params - n_bias_tr:,} head) | dev inference {inf_s:.1f} s | "
      f"peak training memory {train_mem:.2f} GB")
json.dump({"run": RUN_NAME, "build": BUILD, "bitfit_groups": list(BITFIT_GROUPS), "log": log,
           "trainable": tr_params, "total": tot_params, "thresh_mode": THRESH_MODE,
           "final_thr": np.asarray(FINAL_THR).tolist(),
           "dev": {k: res[k] for k in ("f1", "f1_all", "micro_f1", "subset_acc")},
           "dev_probs": {k: v.round(4).tolist() for k, v in dev_probs.items()}},
          open(f"{OUT_DIR}/dev_log.json", "w"))

# Predict test with the best weights and final thresholds -> /kaggle/working/<RUN_NAME>/submission.zip
write_submission(model, BUILD, BATCH_SIZE, AGG, FINAL_THR, MIN_LABELS, best_path=SAVE, out_dir=OUT_DIR)
print("files:", sorted(os.listdir(OUT_DIR)))


pos_weight: min 1.63 median 7.84 max 10.00
  ffn_in    8 tensors   32,768 values  TRAINABLE
  ffn_out   8 tensors    8,192 values  TRAINABLE
  ln1       8 tensors    8,192 values  TRAINABLE
  ln2       8 tensors    8,192 values  TRAINABLE
  ln_f      1 tensors    1,024 values  TRAINABLE
trainable decoder biases: 58,368 | other trainable decoder params: 0
after freezing: max |diff| vs Cell B = 0.00e+00 (should be 0)
trainable 0.161M / total 111.08M (0.14%)


windowing:   0%|          | 0/42750 [00:00<?, ?it/s]

42750 cases -> 156099 windows | windows per case: mean 3.65 max 313


windowing:   0%|          | 0/10181 [00:00<?, ?it/s]

10181 cases -> 41355 windows | windows per case: mean 4.06 max 255


epoch 1/1:   0%|          | 0/39398 [00:00<?, ?it/s]

KeyboardInterrupt: 